# NumJa GPU POC — TornadoVM 5.2.0-jdk21 on NVIDIA T4

Runs `bench.tornadopoc.GemmBench` from branch `gsd/phase-05-hardware-abstraction-layer-gpu-poc`.
Output is labelled key=value so you can paste it into `docs/05-GPU-POC-RESULTS.md`.

**Runtime setup:** Menu > Runtime > Change runtime type > Hardware accelerator = **T4 GPU** > OS = Ubuntu 22.04.

## Step 1 — Verify GPU + JDK
Confirms the runtime actually has the T4 (catches the "selected CPU runtime by mistake" failure mode up front).

In [ ]:
!nvidia-smi | head -20
!echo "---"
!java -version 2>&1
!echo "---"
!javac -version 2>&1

Expected: `nvidia-smi` lists `Tesla T4`; Java 21.x (TornadoVM 5.2.0-jdk21 needs JDK 21).
If JDK is not 21, run Step 1b.

## Step 1b — Install JDK 21 (skip if already 21.x)

In [ ]:
!apt-get update -qq && apt-get install -y -qq openjdk-21-jdk 2>&1 | tail -5
!update-alternatives --list java 2>&1 | head -5
!update-java-alternatives --list 2>&1 | head -5

## Step 2 — Install TornadoVM 5.2.0-jdk21 SDK

Downloads the SDK tarball, extracts to `/opt/tornadovm`, sets `TORNADO_SDK`. The SDK bundles a local Maven repo at `$TORNADO_SDK/share/java/tornadovm-maven-repo/` which is how `mvn package` resolves the `io.github.beehive-lab:tornado-*:5.2.0-jdk21` deps.

In [ ]:
import os, subprocess, urllib.request, tarfile, shutil, glob, sys

TORNADO_VERSION = "5.2.0-jdk21"
SDK_DIR = "/opt/tornadovm"
TARBALL = f"tornadovm-{TORNADO_VERSION}-cuda-linux-amd64.tar.gz"
URL = f"https://github.com/beehive-lab/TornadoVM/releases/download/v{TORNADO_VERSION}/{TARBALL}"

if not os.path.isdir(SDK_DIR):
    print(f"Downloading {URL} ...")
    urllib.request.urlretrieve(URL, f"/tmp/{TARBALL}")
    print("Extracting ...")
    with tarfile.open(f"/tmp/{TARBALL}", "r:gz") as t:
        t.extractall("/opt/")
    extracted = glob.glob("/opt/tornadovm*")
    if len(extracted) == 1 and extracted[0] != SDK_DIR:
        os.rename(extracted[0], SDK_DIR)
    print("Done.")
else:
    print(f"{SDK_DIR} already exists; skipping download.")

os.environ["TORNADO_SDK"] = SDK_DIR
setenv = os.path.join(SDK_DIR, "setenv.sh")
if os.path.isfile(setenv):
    # Source setenv.sh in a subshell, then copy its env vars back
    cmd = f"bash -c 'source {setenv} && env'"
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    for line in proc.stdout.splitlines():
        if "=" in line:
            k, _, v = line.partition("=")
            os.environ[k] = v
    print(f"Sourced {setenv}")
else:
    print(f"WARNING: {setenv} not found")

print("TORNADO_SDK =", os.environ.get("TORNADO_SDK"))
print("PATH contains tornado:", any("tornado" in p.lower() for p in os.environ.get("PATH", "").split(":")))

In [ ]:
# Sanity: tornado device list
!$TORNADO_SDK/bin/tornado --devices 2>&1 | head -20

Expected: a list showing at least `nvidia:0:0` (T4) and CPU devices.
If `nvidia:0:0` is absent, the SDK install failed silently — re-run Step 2.

## Step 3 — Clone the NumJa repo + checkout the Phase 5 branch

In [ ]:
REPO_DIR = "/content/java_ml"
BRANCH = "gsd/phase-05-hardware-abstraction-layer-gpu-poc"

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists; resetting to {BRANCH}")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/minhhhduc/jml.git", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)

print("HEAD:", subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

## Step 4 — Install Maven 3.9.15 (Colab ships with a Maven, but pin to the project's expected version)

In [ ]:
MVN_VERSION = "3.9.15"
MVN_DIR = f"/opt/maven-{MVN_VERSION}"

if not os.path.isdir(MVN_DIR):
    mvn_url = f"https://archive.apache.org/dist/maven/maven-3/{MVN_VERSION}/binaries/apache-maven-{MVN_VERSION}-bin.tar.gz"
    print(f"Downloading {mvn_url} ...")
    urllib.request.urlretrieve(mvn_url, "/tmp/mvn.tgz")
    with tarfile.open("/tmp/mvn.tgz", "r:gz") as t:
        t.extractall("/opt/")
    extracted = glob.glob("/opt/apache-maven*")[0]
    os.rename(extracted, MVN_DIR)
else:
    print(f"{MVN_DIR} already exists.")

MVN = os.path.join(MVN_DIR, "bin", "mvn")
os.environ["PATH"] = os.path.dirname(MVN) + ":" + os.environ.get("PATH", "")
print("Maven:", subprocess.run([MVN, "--version"], capture_output=True, text=True).stdout.splitlines()[0])

## Step 5 — Build the POC module + test the production modules

In [ ]:
# First, install modules/numja to the local repo so bench/tornado-poc can resolve com.numja:numja-core:0.1.0
%cd /content/java_ml
!$MVN -q -pl modules/numja -am install -DskipTests 2>&1 | tail -20

In [ ]:
# Run the full numja test suite first to confirm 47/47 green on Colab
%cd /content/java_ml
!$MVN -pl modules/numja -am test 2>&1 | grep -E "Tests run:|BUILD" | tail -15

In [ ]:
# Build the POC shaded jar (this is the long step — first run pulls TornadoVM SDK jars)
%cd /content/java_ml
!$MVN -q -pl bench/tornado-poc -am package -DskipTests 2>&1 | tail -20

In [ ]:
!ls -la /content/java_ml/bench/tornado-poc/target/tornado-poc-jar.jar 2>&1

If the jar is missing, the build failed. Check the cell above for `Could not resolve dependency` — usually means the SDK's bundled Maven repo isn't on the path. Run this to confirm:

```
!ls $TORNADO_SDK/share/java/tornadovm-maven-repo/io/github/beehive-lab/tornado-api/5.2.0-jdk21/
```

If the SDK bundles its repo elsewhere, set `-Dmaven.repo.local=$TORNADO_SDK/share/java/tornadovm-maven-repo` when running mvn.

## Step 6 — Run the POC (CPU baseline + GPU path on T4)

In [ ]:
JAR = "/content/java_ml/bench/tornado-poc/target/tornado-poc-jar.jar"
DEVICE = "nvidia:0:0"  # T4

# CPU-only baseline run (no -Dtornado.device) — useful for documenting the local box's behaviour on Colab too
!java -cp $JAR -Dbench.env=colab -Dbench.size=4096 bench.tornadopoc.GemmBench 2>&1 | tee /tmp/poc-cpu.log

In [ ]:
# GPU run with explicit device override — the real measurement
!java -cp $JAR -Dtornado.device=$DEVICE -Dbench.env=colab -Dbench.size=4096 bench.tornadopoc.GemmBench 2>&1 | tee /tmp/poc-gpu.log

## Step 7 — Capture output for the decision doc

The key=value lines above are parser-friendly. Copy the GPU run's stdout into `docs/05-GPU-POC-RESULTS.md` under `## Colab (NVIDIA T4)`.

In [ ]:
# Quick verdict summary
import re
with open("/tmp/poc-gpu.log") as f:
    log = f.read()

metrics = {}
for line in log.splitlines():
    m = re.match(r"^([a-z_]+)=(.+)$", line.strip())
    if m:
        metrics[m.group(1)] = m.group(2)

if metrics:
    print("=== Captured GPU metrics ===")
    for k in ["env", "device", "jdk", "hardware", "tornado.device", "size",
             "cpu_baseline_ms", "gpu_ms", "transfer_ms",
             "speedup_ratio", "transfer_pct", "result", "verdict"]:
        if k in metrics:
            print(f"  {k} = {metrics[k]}")
else:
    print("No key=value metrics found in log. Full log:")
    print(log[-2000:])

## Decision rubric

- **GO** if `verdict=GO` (speedup_ratio >= 2.0 AND transfer_pct < 50.0)
- **NO-GO** if `verdict=NO-GO` (either speedup < 2.0, transfer overhead too high, or GPU_ABSENT on a no-GPU box)
- **INSUFFICIENT_DATA** if non-finite values (NaN/Inf — usually means the kernel crashed)

Paste the GPU run's key=value block into `docs/05-GPU-POC-RESULTS.md` under `## Colab (NVIDIA T4)` and add a one-line go/no-go recommendation.